# Function 8b: week-by-week progression, and the move to regression at week 6

You're optimising an eight-dimensional black-box function, where each of the eight input parameters affects the output, but the internal mechanics are unknown. Your objective is to find the parameter combination that maximises the function's output, such as performance, efficiency or validation accuracy. Because the function is high-dimensional and likely complex, global optimisation is hard, so identifying strong local maxima is often a practical strategy. For example, imagine you're tuning an ML model with eight hyperparameters: learning rate, batch size, number of layers, dropout rate, regularisation strength, activation function (numerically encoded), optimiser type (encoded) and initial weight range. Each input set returns a single validation accuracy score. Your goal is to maximise this score.

## How this notebook works

This notebook follows Function 8 **week by week**, in order, from week 0 to week 10 -- at which point the function is assumed to have been maximised. Each week is a section of its own: week 0 has the 40 initial rows and no weekly result, and the result of round `w` only arrives in week `w`.

1. **Data loading:** all Function 8 rounds up to `LAST_WEEK`.
2. **Weeks 0 to 5, the search as it happened:** the weekly queries came from a Gaussian-process-based search with a neural-network cross-check, described stage by stage below and implemented in `f8.ipynb`. Each week's section records the data that existed (the new result and the best `y` so far) The sections do not re-run the search, and no quadratic model is used.
3. **Week 6, a different approach:** in week 6 the problem was treated as a regression problem instead of taking another Gaussian-process step. An ordinary least-squares model with linear and squared terms is fitted to rounds 0 to 5 and its maximum is found analytically. The round-6 result is then compared with the prediction.
4. **Weeks 7 to 10, confirming the maximum:** the later rounds were used as diagnostics, not to search: small deliberate steps around the predicted optimum and a repeat of it. Each is compared with what the week-6 regression predicted.
5. **Retrospective:** the same regression idea replayed week by week with one fixed rule, to ask when the structure could first have been identified.

Everything computed here is derived from `data.csv`. Change `LAST_WEEK` to follow a different span.

**About the code.** The original workbook ran to more than 1,000 lines, with many iterations and diagnostic cells. To make it easier to use, the code was refactored with the help of Claude Code. in so doing significant time was spend rebuilding the pipeline while guarding against hindsighting. 

**Guarding against hindsight.**

- *Rounds 0 to 5* show the data that existed at each round.
- *Round 6* is fitted on rounds 0 to 5 and eventuatted in what appears to be a maximized function.
- **One qualification.** This shows that the regression model reproduces the supplied observations and has a maximum of 10. It does not prove that the unknown black-box function must equal this quadratic everywhere. However, the effective zero residual and the simple coefficient structure are strong evidence that the underlying function was maximised in round-6 result.
- *Rounds 7 to 10* are compared with its predictions to confirm the maximum.

**Data-handling choices.**

- *Precision.* `data.csv` stores `y` at full float64 precision. An exact model fits to about 1e-14.
- *Scope.* This works when the function is low order and separable. It does not apply to noisy or high-order surfaces.
- *Input format.* Proposed queries are printed to 6 decimals and clipped to [0, 0.999999], as the portal requires.

In [27]:
# =============================================================================
# IMPORTS AND SETTINGS
# =============================================================================

import itertools
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

DATA_PATH = "../data/data.csv"
FUNCTION_ID = 8
LAST_WEEK = 10           # follow weeks 0 .. LAST_WEEK (week 0 = initial design only)
RETRO_LAST_WEEK = 6      # the retrospective replays weeks 0 .. RETRO_LAST_WEEK

UPPER_BOUND = 0.999999   # portal input format: 6 decimals, inputs in [0, 0.999999]

In [28]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[(data["function"] == FUNCTION_ID) & (data["round"] <= LAST_WEEK)].copy()
function_data = function_data.sort_values("round", kind="stable").reset_index(drop=True)

feature_cols = [
    f"x{i}" for i in range(1, 9)
    if function_data[f"x{i}"].notna().any()
]
d = len(feature_cols)

print(f"Function {FUNCTION_ID}: {d} inputs, rounds 0-{LAST_WEEK}")
print("Observations per round:", function_data.groupby("round").size().to_dict())

def rows_up_to(week):
    """The data that exists in `week`: rounds 0 .. week and nothing later."""
    train = function_data[function_data["round"] <= week]
    return train[feature_cols].to_numpy(), train["y"].to_numpy(), train["round"].to_numpy()

def format_point(x):
    """Format a point as the dash-separated six-decimal submission string."""
    return "-".join(f"{v:.6f}" for v in np.clip(x, 0.0, UPPER_BOUND))

def search_view(week):
    """Week `week`: the new result and the best y so far."""
    X, y, _ = rows_up_to(week)
    n = len(y)

    print("=" * 96)
    print(f"WEEK {week}" + (": initial design only, no weekly result yet" if week == 0 else f": round-{week} result"))
    print("=" * 96)

    if week == 0:
        print(f"Rows: {n}")
        print(f"y range: {y.min():.3f} to {y.max():.6f} | mean {y.mean():.3f} | SD {y.std():.3f}")
    else:
        before = y[:-1]
        incumbent_before = X[np.argmax(before)]
        print(f"Submitted input : {format_point(X[-1])}")
        print(f"Observed y      : {y[-1]:.6f}")
        best = y.max()
        status = "new best" if y[-1] > before.max() else "no new best"
        print(f"Best observed so far: {best:.6f} ({status}; change {best - before.max():+.6f})")
        print(f"Euclidean distance of this input from the previous best input: {np.linalg.norm(X[-1] - incumbent_before):.3f}")

Function 8: 8 inputs, rounds 0-10
Observations per round: {0: 40, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1}


## Weeks 0 to 5: Modelling and optimisation

During this period, the weekly submissions were selected primarily using Gaussian Process (GP) models, with neural networks as an additional check. The analysis was documented in `f8.ipynb` was developed from an earlier workbook. It containing very extensive local-geometry and kernel-comparison diagnostics. Much of these diagnostics (which were heursitc and part of a learning process) were removed and/or simplified before final submission of this notebook.

The analysis below was undertaken when preparing the round-5 submission. At that point, 44 observations were available through round 4, and the best result remained the round-3 observation (y = 9.955). The process involved several stages:

1. **Local geometry (no model).** Examined the distances between the best recent observations in eight-dimensional space, identified historical points closest to the incumbent, and assessed whether high objective values were concentrated in the same region. The analysis also considered the direction of recent changes and the density of observations around the incumbent.

2. **Conditional profiles across GP kernels.** Fitted Gaussian Processes using RBF, Matérn 3/2 and Matérn 5/2 kernels with automatic relevance determination (ARD). Each model was evaluated using leave-one-out cross-validation before being fitted to the full dataset. Individual inputs were then varied around the incumbent holding the remaining inputs constant. This helped inform whether agreement or otherwise between different kernels for local improvement.

3. **Tuned ARD Gaussian Process.** Fitted a Matérn 5/2 ARD model using standardised inputs and a Yeo-Johnson transformation of the target variable. The noise parameter was selected through a leave-one-out grid search. Predictive performance was strong, with a cross-validated R² of approximately 0.99.

4. **Multivariate local trust-region search.** The analysis was extended to consider simultaneous changes in multiple inputs. Sobol sampling was used to generate candidates around the incumbent, with wider ranges assigned to inputs showing stronger improvement signals. Candidates were evaluated using Matérn 5/2 and Matérn 3/2 models. The analysis focused on agreement between the models and whether the earlier one-dimensional results remained consistent when inputs were varied together.

5. **Conservative refinement.** The search was then narrowed to a region supported by both GP models. A consensus score was developed to account for differences between model predictions, distance from the incumbent and predictive uncertainty. Both conservative and more aggressive candidates were considered.

6. **Neural-network cross-check.** To support learning and as a validation step, a small regularised multilayer perceptron (MLP) was tested against the GP models. The initial cross-check produced an R² of approximately 0.76, compared with approximately 0.99 for the GP. The neural-network ensemble also showed little variation in its predictions around the incumbent, making it less useful as the primary surrogate. A subsequent evaluation, with scaling performed separately within each cross-validation fold and ten random seeds, improved the R² to approximately 0.95. The neural network was therefore regarded and retained as an additional diagnostic.

**Weekly results.** The sections below record the observations received in each round and the best objective value achieved up to that point. These sections document the available data rather than repeating the optimisation process described above.

### Week 0

In [29]:
search_view(0)

WEEK 0: initial design only, no weekly result yet
Rows: 40
y range: 5.592 to 9.598482 | mean 7.815 | SD 0.947


### Week 1

In [30]:
search_view(1)

WEEK 1: round-1 result
Submitted input : 0.080166-0.217529-0.000000-0.057788-0.928887-0.287648-0.000000-0.447904
Observed y      : 9.799737
Best observed so far: 9.799737 (new best; change +0.201255)
Euclidean distance of this input from the previous best input: 1.000


### Week 2

In [31]:
search_view(2)

WEEK 2: round-2 result
Submitted input : 0.000000-0.496823-0.000000-0.384575-0.696280-0.634875-0.143137-0.745253
Observed y      : 9.721842
Best observed so far: 9.799737 (no new best; change +0.000000)
Euclidean distance of this input from the previous best input: 0.689


### Week 3

In [32]:
search_view(3)

WEEK 3: round-3 result
Submitted input : 0.072114-0.141719-0.097856-0.235510-0.827528-0.525530-0.074018-0.648656
Observed y      : 9.954954
Best observed so far: 9.954954 (new best; change +0.155217)
Euclidean distance of this input from the previous best input: 0.400


### Week 4

In [33]:
search_view(4)

WEEK 4: round-4 result
Submitted input : 0.068722-0.042450-0.094771-0.313902-0.813079-0.631761-0.097470-0.525118
Observed y      : 9.916857
Best observed so far: 9.954954 (no new best; change +0.000000)
Euclidean distance of this input from the previous best input: 0.208


### Week 5

In [34]:
search_view(5)

WEEK 5: round-5 result
Submitted input : 0.026789-0.125152-0.153668-0.029718-0.999976-0.571921-0.103230-0.674846
Observed y      : 9.928058
Best observed so far: 9.954954 (no new best; change +0.000000)
Euclidean distance of this input from the previous best input: 0.285


In [49]:
# =============================================================================
# WEEKS 0 TO 5 SUMMARY TABLE
# =============================================================================

rows = []
best_so_far = -np.inf
for w in range(0, 6):
    X, y, r = rows_up_to(w)
    new_y = y.max() if w == 0 else y[-1]
    improved = new_y > best_so_far
    best_so_far = max(best_so_far, new_y)
    rows.append({"Week": w, "Rows": len(y), "Result": new_y if w else np.nan,
                 "Best observed so far": best_so_far, "New best": "yes" if (w and improved) else ""})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

 Week  Rows  Result  Best observed so far New best
    0    40     NaN                9.5985         
    1    41  9.7997                9.7997      yes
    2    42  9.7218                9.7997         
    3    43  9.9550                9.9550      yes
    4    44  9.9169                9.9550         
    5    45  9.9281                9.9550         


## Week 6: pivot to regression

In week 6 the problem was approached differently. Instead of another Gaussian-Process step, a number of other algorithms and tools were used. This includeed treating the solution as a deterministic **regression problem**. The code below uses observations from weeks 0 to 5 (45 rows), fits an OLS quadratic model without interaction terms, and finds its maximum analytically.

The regression model has 17 coefficients estimated from 45 observations. The steps were:

1. Load the week 0 to 5 observations.
2. Fit the quadratic regression by ordinary least squares.
3. Report the coefficients, R², RMSE and maximum residual.
4. Recover the function and calculate the optimal inputs from the fitted coefficients.
5. Verify the predicted maximum within the bounds [0, 1].

In [36]:
# =============================================================================
# WEEK 6, STEPS 1 AND 2: LOAD ROUNDS 0-5 AND FIT THE QUADRATIC REGRESSION
# =============================================================================
# Linear and squared terms for each input; no interaction terms.

variables = feature_cols
X, y, _ = rows_up_to(5)
print(f"Observations: {len(y)} (rounds 0-5)")
print(f"Input variables: {X.shape[1]}")
print(function_data[function_data["round"] <= 5].groupby("round").size().to_string())

X_quad = np.column_stack([X, X ** 2])

model = LinearRegression()
model.fit(X_quad, y)

intercept = model.intercept_
linear_coef = model.coef_[:8]
quadratic_coef = model.coef_[8:]

Observations: 45 (rounds 0-5)
Input variables: 8
round
0    40
1     1
2     1
3     1
4     1
5     1


In [37]:
# =============================================================================
# WEEK 6, STEP 3: REGRESSION RESULTS
# =============================================================================

y_pred = model.predict(X_quad)

print(f"Intercept: {intercept:.10f}")
print(f"R-squared: {r2_score(y, y_pred):.12f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, y_pred)):.3e}")
print(f"Maximum absolute error: {np.max(np.abs(y - y_pred)):.3e}")

print("\nEstimated coefficients:")
print(pd.DataFrame({"Variable": variables, "Linear coefficient": linear_coef,
                    "Quadratic coefficient": quadratic_coef}).to_string(index=False))

Intercept: 9.1983000000
R-squared: 1.000000000000
RMSE: 1.685e-15
Maximum absolute error: 3.553e-15

Estimated coefficients:
Variable  Linear coefficient  Quadratic coefficient
      x1                0.40                   -2.0
      x2                0.30                   -1.0
      x3                0.78                   -3.0
      x4                0.30                   -1.0
      x5                0.80                   -0.5
      x6                1.00                   -1.0
      x7                0.80                   -2.0
      x8                0.12                   -0.1


In [38]:
# =============================================================================
# WEEK 6, STEPS 4 AND 5: RECOVER THE FUNCTION AND FIND THE MAXIMUM
# =============================================================================
# f_i(x_i) = a_i x_i + b_i x_i^2. First-order condition: a_i + 2 b_i x_i = 0,
# so x_i* = -a_i / (2 b_i). If every b_i is negative the model is strictly
# concave and this stationary point is the unique global maximum of the fitted model.

print("RECOVERED FUNCTION")
print(f"f(x) = {intercept:.6f}")
for variable, a, b in zip(variables, linear_coef, quadratic_coef):
    print(f"       {a:+.6f}*{variable} {b:+.6f}*{variable}^2")

all_concave = bool(np.all(quadratic_coef < 0))
print(f"\nAll quadratic coefficients negative (strictly concave): {all_concave}")
if not all_concave:
    print("The analytic maximum requires further checks; no optimum is calculated.")
else:
    x_opt = np.clip(-linear_coef / (2 * quadratic_coef), 0, 1)     # enforce the input bounds [0, 1]
    y_opt = model.predict(np.concatenate([x_opt, x_opt ** 2]).reshape(1, -1))[0]

    print("\nOPTIMAL INPUTS")
    for name, value in zip(variables, x_opt):
        print(f"{name}: {value:.6f}")
    print(f"\nPredicted maximum: {y_opt:.10f}")

    best_observed = y.max()
    print(f"Best observed output (rounds 0-5): {best_observed:.6f}")
    print(f"Improvement predicted: {y_opt - best_observed:+.6f}")
    print(f"\nProposed query for round 6: {format_point(x_opt)}")

RECOVERED FUNCTION
f(x) = 9.198300
       +0.400000*x1 -2.000000*x1^2
       +0.300000*x2 -1.000000*x2^2
       +0.780000*x3 -3.000000*x3^2
       +0.300000*x4 -1.000000*x4^2
       +0.800000*x5 -0.500000*x5^2
       +1.000000*x6 -1.000000*x6^2
       +0.800000*x7 -2.000000*x7^2
       +0.120000*x8 -0.100000*x8^2

All quadratic coefficients negative (strictly concave): True

OPTIMAL INPUTS
x1: 0.100000
x2: 0.150000
x3: 0.130000
x4: 0.150000
x5: 0.800000
x6: 0.500000
x7: 0.200000
x8: 0.600000

Predicted maximum: 10.0000000000
Best observed output (rounds 0-5): 9.954954
Improvement predicted: +0.045046

Proposed query for round 6: 0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000


In [50]:
# =============================================================================
# WEEK 6: ROUND-6 RESULT
# =============================================================================
# Fit the quadratic model using observations from Rounds 0–5.
# Round 6 is excluded from fitting and used only to validate
# the predicted optimum against the actual observed result.

round6 = function_data[function_data["round"] == 6]
x6, y6 = round6[feature_cols].to_numpy()[0], float(round6["y"].iloc[0])
print(f"Submitted input : {format_point(x6)}")
print(f"Observed y      : {y6:.6f}")
if all_concave:
    print(f"Matches the proposed query: {bool(np.allclose(x6, x_opt, atol=1e-6))} "
          f"(largest difference in any input {np.abs(x6 - x_opt).max():.1e})")
    print(f"Predicted y     : {y_opt:.6f} (error {abs(y6 - y_opt):.1e})")

Submitted input : 0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000
Observed y      : 10.000000
Matches the proposed query: True (largest difference in any input 1.3e-14)
Predicted y     : 10.000000 (error 1.8e-15)


### Why this works

The key is choosing an appropriate regression specification.

- A standard linear regression would miss the curvature.
- A full quadratic model with interactions would need 45 coefficients, equal to the number of observations, leaving no spare equations to judge the fit.
- The quadratic model needs only 17 coefficients and fits all 45 observations.
- Because every estimated quadratic coefficient is negative, the fitted objective is concave, and its stationary point is the unique global maximum of the fitted model over the permitted input domain.

## Weeks 7 to 10: confirming the maximum

The regression predicted a maximum of 10 at one interior point, and round 6 reached it. Rounds 7 to 10 were not used to search further. They were diagnostics, run to check that this is really a maximum:

- small deliberate steps away from the optimum, to see whether the function falls as the concave model predicts;
- a repeat of the optimum, to see whether the same input gives the same output.

Each result is compared with the week-6 model (fitted on rounds 0 to 5 only), so these are out-of-sample forecasts.

In [40]:
# =============================================================================
# WEEKS 7 TO 10: HELPER FOR THE CONFIRMATION ROUNDS
# =============================================================================
# At an interior maximum the gradient is zero, so a step of size offset changes
# a separable quadratic by exactly sum(quadratic_coef * offset^2), which is
# negative when every quadratic coefficient is negative.

confirm_rows = []


def probe_report(week):
    """Round `week` as a diagnostic around the predicted maximum."""
    new = function_data[function_data["round"] == week]
    x_new, y_new = new[feature_cols].to_numpy()[0], float(new["y"].iloc[0])
    offset = x_new - x_opt
    predicted = float(model.predict(np.concatenate([x_new, x_new ** 2]).reshape(1, -1))[0])
    predicted_change = float((quadratic_coef * offset ** 2).sum())

    print("=" * 96)
    print(f"WEEK {week}: round-{week} result (diagnostic around the maximum)")
    print("=" * 96)
    print(f"Submitted input : {format_point(x_new)}")
    print("Offset from the optimum: " + "  ".join(f"{c} {round(o, 6) + 0.0:+.3f}" for c, o in zip(feature_cols, offset)))
    print(f"Observed y      : {y_new:.6f} (change from the predicted maximum {y_new - y_opt:+.2e})")
    print(f"Week-6 model forecast: {predicted:.6f} (error {abs(y_new - predicted):.1e})")
    print(f"Change predicted by the concave quadratic alone: {predicted_change:+.2e}")
    if np.abs(offset).max() < 1e-9:
        round6_y = float(function_data[function_data["round"] == 6]["y"].iloc[0])
        print(f"Repeat of the round-6 input: observed y equals round 6's ({round6_y:.6f}): {y_new == round6_y}")
    confirm_rows.append({"Round": week, "Largest offset": np.abs(offset).max(), "Observed y": y_new,
                         "Forecast y": predicted, "Forecast error": abs(y_new - predicted),
                         "Observed change vs 10": y_new - y_opt})

### Week 7

In [41]:
probe_report(7)

WEEK 7: round-7 result (diagnostic around the maximum)
Submitted input : 0.105000-0.155000-0.135000-0.155000-0.805000-0.505000-0.205000-0.605000
Offset from the optimum: x1 +0.005  x2 +0.005  x3 +0.005  x4 +0.005  x5 +0.005  x6 +0.005  x7 +0.005  x8 +0.005
Observed y      : 9.999735 (change from the predicted maximum -2.65e-04)
Week-6 model forecast: 9.999735 (error 0.0e+00)
Change predicted by the concave quadratic alone: -2.65e-04


### Week 8

In [42]:
probe_report(8)

WEEK 8: round-8 result (diagnostic around the maximum)
Submitted input : 0.095000-0.145000-0.125000-0.145000-0.795000-0.495000-0.195000-0.595000
Offset from the optimum: x1 -0.005  x2 -0.005  x3 -0.005  x4 -0.005  x5 -0.005  x6 -0.005  x7 -0.005  x8 -0.005
Observed y      : 9.999735 (change from the predicted maximum -2.65e-04)
Week-6 model forecast: 9.999735 (error 0.0e+00)
Change predicted by the concave quadratic alone: -2.65e-04


### Week 9

In [43]:
probe_report(9)

WEEK 9: round-9 result (diagnostic around the maximum)
Submitted input : 0.095000-0.155000-0.128000-0.150000-0.820000-0.500000-0.200000-0.600000
Offset from the optimum: x1 -0.005  x2 +0.005  x3 -0.002  x4 +0.000  x5 +0.020  x6 +0.000  x7 +0.000  x8 +0.000
Observed y      : 9.999713 (change from the predicted maximum -2.87e-04)
Week-6 model forecast: 9.999713 (error 1.8e-15)
Change predicted by the concave quadratic alone: -2.87e-04


### Week 10

Round-6 input is submitted again.

In [44]:
probe_report(10)

WEEK 10: round-10 result (diagnostic around the maximum)
Submitted input : 0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000
Offset from the optimum: x1 +0.000  x2 +0.000  x3 +0.000  x4 +0.000  x5 +0.000  x6 +0.000  x7 +0.000  x8 +0.000
Observed y      : 10.000000 (change from the predicted maximum +1.78e-15)
Week-6 model forecast: 10.000000 (error 1.8e-15)
Change predicted by the concave quadratic alone: -1.79e-29
Repeat of the round-6 input: observed y equals round 6's (10.000000): True


### Summary of the confirmation rounds

In [45]:
# =============================================================================
# WEEKS 7 TO 10 IN ONE TABLE
# =============================================================================

fmt = lambda v: f"{v:.2e}" if (abs(v) < 1e-3 and v != 0) else f"{v:.6f}"
print(pd.DataFrame(confirm_rows).to_string(index=False, float_format=fmt))
confirm = pd.DataFrame(confirm_rows)
print(f"\nLargest forecast error across rounds 7-10: {confirm['Forecast error'].max():.1e}")
print(f"Highest y observed in these rounds: {confirm['Observed y'].max():.6f} (predicted maximum {y_opt:.6f})")
y7 = next(r["Observed y"] for r in confirm_rows if r["Round"] == 7)
y8 = next(r["Observed y"] for r in confirm_rows if r["Round"] == 8)
print(f"Rounds 7 and 8 were equal and opposite steps and returned identical values ({y7:.6f} and {y8:.6f}): "
      f"{bool(np.isclose(y7, y8, atol=1e-12))}, as a quadratic with its peak at this point requires.")

 Round  Largest offset  Observed y  Forecast y  Forecast error  Observed change vs 10
     7        0.005000    9.999735    9.999735        0.000000              -2.65e-04
     8        0.005000    9.999735    9.999735        0.000000              -2.65e-04
     9        0.020000    9.999713    9.999713        1.78e-15              -2.87e-04
    10        1.27e-14   10.000000   10.000000        1.78e-15               1.78e-15

Largest forecast error across rounds 7-10: 1.8e-15
Highest y observed in these rounds: 10.000000 (predicted maximum 10.000000)
Rounds 7 and 8 were equal and opposite steps and returned identical values (9.999735 and 9.999735): True, as a quadratic with its peak at this point requires.


## Retrospective: When could the structure have been identified?

The regression approach was first tested in week 6. The earlier observations were subsequently re-examined to determine whether the underlying structure could have been identified sooner.

The same regression method and validation criteria were applied to the data available at the end of each week. This retrospective analysis was conducted with the benefit of hindsight and was kept separate from the original optimisation process described above.

Note this code below was developed exclusively using Claude Code. 

**Rule (applied identically to every week).** A week counts as *identified* when all of these hold:

- The selected structure (the simplest within `SELECTION_TOLERANCE` of the best leave-one-out RMSE) has a leave-one-out maximum error below `IDENTIFIED_LOO_TOL`.
- It has at least `MIN_SPARE_ROWS` more rows than parameters, so there are spare equations to check it.
- It is the linear-plus-squares form, so the algebraic solution applies.

### Findings

The retrospective analysis showed that Function 8 could have been solved using the initial 40 observations available in week 0, provided the underlying function:
1. Could be adequately represented by linear and squared terms for each of the eight inputs. Meaning without interactions or higher-order terms.
2. Was deterministic with no material observation noise.
3. The function remained stationary between rounds.
4. The quadratic equation (fitted to the 40 observations) was assumed to describe the entire function.

Initial examination of the squared terms suggested a nonlinear relationship. Seven of the eight individual squared coefficients were negative. The sign test (p = 0.0703) did not reach the conventional 5% significance threshold. The F-test provided much stronger evidence for including quadratic terms, with F(8, 23) = 1.39 × 10²⁷ and p = 5.68 × 10⁻³⁰⁵.

The comparison of alternative model specifications was far more conclusive. The linear model achieved an R² of 0.8990, with a leave-one-out (LOO) RMSE of 0.4049. Importantly, adding squared terms increased R² to 1.0000 and reduced LOO RMSE to an insignificant level of 6.42 × 10⁻¹⁵.

Importantly, the same model was identified at every subsequent week confirming noise assumptions with maximum LOO prediction error remaining insignificant, hence the calculated optimum below remained unchanged.

0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000
Predicted objective value: 10.000000

At the end of week 0, the best observed value was 9.5985, leaving a gap of 0.4015 to the theoretical maximum. By week 3, GP-based optimisation had improved the incumbent to approximately 9.9550, but the quadratic model could already have identified the maximum without additional evaluations.

The main finding was that the GP provided accurate predictions and helped locate a high-performing region. However, it was neither necessary nor the most efficient approach to solving Function 8. A relatively simple quadratic regression model under certain assumptions captured the structure and identified the optimum. The process also demonstrated the value of testing other (perhaps simpler) functional forms before proceeding with more complex surrogate models, particularly when evaluations are limited.

In [46]:
# =============================================================================
# RETROSPECTIVE: SETTINGS AND HELPERS
# =============================================================================

SELECTION_TOLERANCE = 10.0   # choose the simplest model within this factor of the best leave-one-out RMSE
MIN_SPARE_ROWS = 5           # an identified model needs this many more rows than parameters
IDENTIFIED_LOO_TOL = 1e-6    # ... and a leave-one-out maximum error below this
MIN_TEST_ROWS = 2            # a candidate is testable only with this many more rows than parameters


def squares_design(X):
    """Constant, linear terms and squared terms."""
    return np.column_stack([np.ones(len(X)), *X.T, *(X ** 2).T])

def candidate_models(X, y):
    """Candidate structures: (design matrix, fitted target, back-transform)."""
    lin = np.column_stack([np.ones(len(X)), *X.T])
    sq = squares_design(X)
    cubes = np.column_stack([sq, *(X ** 3).T])
    cross = np.column_stack([lin, *[X[:, i] * X[:, j] for i, j in itertools.combinations(range(d), 2)]])
    identity = lambda v: v
    return {
        "Linear": (lin, y, identity),
        "Linear + squares": (sq, y, identity),
        "Linear + squares + cubes": (cubes, y, identity),
        "Linear + pairwise interactions": (cross, y, identity),
        "Linear in log y": (lin, np.log(y), np.exp),
    }

def loo_errors(A, target, back, y):
    """Leave-one-out prediction errors on the y scale."""
    n = len(y)
    errors = []
    for k in range(n):
        keep = np.arange(n) != k
        coef = np.linalg.lstsq(A[keep], target[keep], rcond=None)[0]
        errors.append(y[k] - back(A[k] @ coef))
    return np.array(errors)

def select_structure(battery, tolerance=SELECTION_TOLERANCE):
    """Simplest testable structure within `tolerance` of the best leave-one-out RMSE."""
    testable = battery[battery["Testable"]]
    eligible = testable[testable["LOO RMSE"] <= tolerance * testable["LOO RMSE"].min()]
    return eligible.sort_values(["Parameters", "LOO RMSE"]).iloc[0]

def analyse(week):
    """Apply the fixed rule using only rounds 0 .. week."""
    Xw, yw, _ = rows_up_to(week)
    n = len(yw)

    rows = []
    for name, (A, target, back) in candidate_models(Xw, yw).items():
        p = A.shape[1]
        if n < p + MIN_TEST_ROWS:
            rows.append({"Model": name, "Parameters": p, "Testable": False})
            continue
        coef = np.linalg.lstsq(A, target, rcond=None)[0]
        err = loo_errors(A, target, back, yw)
        rows.append({"Model": name, "Parameters": p, "Testable": True,
                     "R²": 1 - ((yw - back(A @ coef)) ** 2).sum() / ((yw - yw.mean()) ** 2).sum(),
                     "LOO RMSE": np.sqrt((err ** 2).mean()), "LOO max error": np.abs(err).max()})
    battery = pd.DataFrame(rows)
    chosen = select_structure(battery)

    out = {"week": week, "n": n, "battery": battery, "selected": chosen["Model"],
           "params": int(chosen["Parameters"]), "loo_max": chosen["LOO max error"], "incumbent": yw.max()}
    out["identified"] = bool(chosen["Model"] == "Linear + squares"
                             and chosen["LOO max error"] <= IDENTIFIED_LOO_TOL
                             and n - chosen["Parameters"] >= MIN_SPARE_ROWS)
    if chosen["Model"] == "Linear + squares":
        coef = np.linalg.lstsq(squares_design(Xw), yw, rcond=None)[0]
        b, a = coef[1:1 + d], -coef[1 + d:]
        out.update(coef=coef, b=b, a=a, concave=bool((a > 0).all()))
        if out["concave"]:
            out.update(x_star=b / (2 * a), y_star=coef[0] + (b ** 2 / (4 * a)).sum())
    return out

In [47]:
# =============================================================================
# RETROSPECTIVE, WEEK 0: WHAT THE INITIAL DESIGN ALONE SHOWS
# =============================================================================
# Only the 40 initial rows. Is the missing structure curvature, and which
# structure does the fixed rule select?

X0, y0, _ = rows_up_to(0)
n0 = len(y0)
A_lin0 = np.column_stack([np.ones(n0), *X0.T])
res_lin0 = y0 - A_lin0 @ np.linalg.lstsq(A_lin0, y0, rcond=None)[0]

# Curvature in the linear residual: one input at a time, and a nested-model test
rows = []
for i, name in enumerate(feature_cols):
    A = np.column_stack([np.ones(n0), X0[:, i], X0[:, i] ** 2])
    coef = np.linalg.lstsq(A, res_lin0, rcond=None)[0]
    resid = res_lin0 - A @ coef
    se = np.sqrt((resid ** 2).sum() / (n0 - 3) * np.linalg.inv(A.T @ A)[2, 2])
    rows.append({"Input": name, "Square coef": coef[2], "t (one at a time)": coef[2] / se})
curvature = pd.DataFrame(rows)
print(curvature.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
n_negative = int((curvature["Square coef"] < 0).sum())
print(f"\nSquare coefficients negative: {n_negative} of {d} (sign test p = {stats.binomtest(n_negative, d, 0.5).pvalue:.4f})")

A_sq0 = squares_design(X0)
rss_lin = (res_lin0 ** 2).sum()
rss_sq = ((y0 - A_sq0 @ np.linalg.lstsq(A_sq0, y0, rcond=None)[0]) ** 2).sum()
df_den = n0 - A_sq0.shape[1]
F = ((rss_lin - rss_sq) / d) / (rss_sq / df_den)
print(f"Nested F-test, linear vs linear + squares: F({d}, {df_den}) = {F:.3g}, p = {stats.f.sf(F, d, df_den):.3g}")
print("The per-input t-statistics depend on how the test is specified; the nested comparison and the structure comparison below do not.")

# Structure comparison and selection
r0 = analyse(0)
fmt = lambda v: f"{v:.3e}" if (abs(v) < 1e-2 and v != 0) else f"{v:.4f}"
print("\n" + r0["battery"].to_string(index=False, float_format=fmt))
print(f"\nSelected by the fixed rule: {r0['selected']} ({r0['params']} parameters, {r0['n'] - r0['params']} spare rows)")
tolerances = [1.0, 1.05, 2, 10, 1e3, 1e6, 1e9, 1e12, 1e14]
stable = [t for t in tolerances if select_structure(r0["battery"], t)["Model"] == r0["selected"]]
print(f"Selection sensitivity: '{r0['selected']}' is chosen for tolerances {min(stable):g} to {max(stable):g}")
print(f"Identified under the fixed rule: {'yes' if r0['identified'] else 'no'}")
if "x_star" in r0:
    print(f"Predicted maximum y* = {r0['y_star']:.6f}; proposal from week 0: {format_point(r0['x_star'])}")

Input  Square coef  t (one at a time)
   x1       -1.764             -3.151
   x2       -1.054             -1.722
   x3       -2.615             -5.125
   x4       -1.936             -2.442
   x5        0.322              0.419
   x6       -1.845             -2.852
   x7       -0.481             -0.655
   x8       -0.646             -0.963

Square coefficients negative: 7 of 8 (sign test p = 0.0703)
Nested F-test, linear vs linear + squares: F(8, 23) = 1.39e+27, p = 5.68e-305
The per-input t-statistics depend on how the test is specified; the nested comparison and the structure comparison below do not.

                         Model  Parameters  Testable     R²  LOO RMSE  LOO max error
                        Linear           9      True 0.8990    0.4049         1.0597
              Linear + squares          17      True 1.0000 6.417e-15      2.309e-14
      Linear + squares + cubes          25      True 1.0000 8.506e-15      2.309e-14
Linear + pairwise interactions          37      T

In [48]:
# =============================================================================
# RETROSPECTIVE: THE FIXED RULE APPLIED TO EVERY WEEK, 0 TO RETRO_LAST_WEEK
# =============================================================================
# Each week's model forecasts the next result before it is used; forecast
# errors are out-of-sample.

state = {w: analyse(w) for w in range(RETRO_LAST_WEEK + 1)}
final_x = state[RETRO_LAST_WEEK].get("x_star")

rows = []
for w, r in state.items():
    forecast_error = np.nan
    prev = state.get(w - 1)
    if w > 0 and prev is not None and "coef" in prev:
        new = function_data[function_data["round"] == w]
        pred = float(squares_design(new[feature_cols].to_numpy())[0] @ prev["coef"])
        forecast_error = abs(float(new["y"].iloc[0]) - pred)
    rows.append({"Week": w, "Rows": r["n"], "Identified": "yes" if r["identified"] else "no",
                 "LOO max error": r["loo_max"], "Predicted max y": r.get("y_star", np.nan),
                 "Best observed y": r["incumbent"], "Gap": r.get("y_star", np.nan) - r["incumbent"],
                 "Forecast error of this week's result": forecast_error,
                 "Proposal vs final week": np.abs(r["x_star"] - final_x).max() if "x_star" in r and final_x is not None else np.nan})
fmt = lambda v: f"{v:.2e}" if (abs(v) < 1e-3 and v != 0) else f"{v:.4f}"
print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))

first = next((w for w, r in state.items() if r["identified"]), None)
if first is None:
    print("\nThe fixed rule does not identify the function within the weeks followed.")
else:
    print(f"\nFirst week the fixed rule identifies the function: week {first} "
          f"({'initial design only' if first == 0 else f'rounds 0-{first}'}).")
    print(f"Proposed query then: {format_point(state[first]['x_star'])}")

 Week  Rows Identified  LOO max error  Predicted max y  Best observed y       Gap  Forecast error of this week's result  Proposal vs final week
    0    40        yes       2.31e-14          10.0000           9.5985    0.4015                                   NaN                2.41e-13
    1    41        yes       2.40e-14          10.0000           9.7997    0.2003                              1.07e-14                1.71e-14
    2    42        yes       2.75e-14          10.0000           9.7997    0.2003                              1.78e-15                1.24e-14
    3    43        yes       4.71e-14          10.0000           9.9550    0.0450                              1.07e-14                4.93e-14
    4    44        yes       4.62e-14          10.0000           9.9550    0.0450                              1.78e-15                2.85e-14
    5    45        yes       3.73e-14          10.0000           9.9550    0.0450                              3.73e-14                3